<a href="https://colab.research.google.com/github/ripims1/COMP-3608-PROJECT/blob/tyler-data-blending/Blended_Models_NN_%2B_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================
# Logistic Regression
# ===================

import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# Load the modified datasets
#These files were created in the preprocessing step where we cleaned and encoded the raw data.
trainData = pd.read_csv('train_processed.csv')
testData = pd.read_csv('test_processed.csv')

X_train = trainData.drop('Churn', axis=1)   #contains all the columns the model will learn from, except the target variable 'Churn'
y_train = trainData['Churn'] #contains the target variable 'Churn' which indicates whether a customer churned (1) or not (0)

# Train the logistic regression model
#max_iter = 5000 to give the model enough iterations to find the best coefficients for all the features.
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

predictions = model.predict_proba(testData)[:, 1] #predict_proba gives the probability of each class (churn or not churn) for each customer in the test set.
print(predictions)

# Plot the Histogram of Predicted Churn Probabilities
threshold = 0.5 #This is the cutoff point we will use to classify customers as likely to churn (1) or not (0).

plt.figure(figsize=(8, 5))
plt.hist(predictions, bins=50, color='darkblue', edgecolor='black')
plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold ({threshold})')
plt.title('Predicted Churn Probabilities')
plt.xlabel('Probability of Churn')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

#Pulls the coefficients from the trained logistic regression model
# and identifies the top 15 features that have the most impact on predicting customer churn.
# It then creates a horizontal bar chart to visualize these features, coloring the bars red for
# features that increase churn risk and blue for those that decrease it.

# Get the coefficient for each feature
# Positive = increases churn risk, Negative = decreases churn risk
coefficients = pd.Series(model.coef_[0], index=X_train.columns)

# Sort by absolute value to find the most impactful features
top15 = coefficients.abs().nlargest(15).index

# Get the actual coefficient values for those top 15 features
top15_coefs = coefficients[top15].sort_values()

# Color the bars based on whether they increase or decrease churn
#Red = increases churn risk (positive coefficient)
#Blue = decreases churn risk (negative coefficient)
bar_colors = []
for value in top15_coefs:
    if value > 0:
        bar_colors.append('maroon')   # increases churn risk
    else:
        bar_colors.append('darkblue') # decreases churn risk

# Plot the top 15 features with a horizontal bar chart
plt.figure(figsize=(9, 6))
plt.barh(top15_coefs.index, top15_coefs.values, color=bar_colors, edgecolor='black')
plt.axvline(x=0, color='black', linestyle='--')
plt.title('Top 15 Features That Influence Churn')
plt.xlabel('Coefficient Value (red = more churn, blue = less churn)')
plt.tight_layout()
plt.show()

# This section calculates the churn rate for each category in the dataset and visualizes it with a bar chart.

numeric_columns = ['tenure', 'MonthlyCharges', 'TotalCharges'] # These are the numeric columns in the dataset that we will exclude when looking for category columns.

# Find all the category columns (they only contain 0s and 1s)
category_columns = []
for col in X_train.columns:
    if col not in numeric_columns and X_train[col].nunique() == 2:
        category_columns.append(col)

# For each category, calculate the churn rate of customers in that category
churn_rates = {}
for col in category_columns:
    # Find all customers who belong to this category
    customers_in_category = X_train[col] == 1

    # Skip if no customers belong to this category
    if customers_in_category.sum() == 0:
        continue
    # Calculate what percentage of those customers churned
    churn_rates[col] = y_train[customers_in_category].mean() * 100

# Put results into a dataframe and sort highest to lowest
churn_df = pd.DataFrame({
    'Category': list(churn_rates.keys()),
    'ChurnRate': list(churn_rates.values())
})
churn_df = churn_df.sort_values('ChurnRate', ascending=False).reset_index(drop=True)

# Calculate the overall churn rate to use as a reference line
overall_churn_rate = y_train.mean() * 100

# Colour each bar based on whether the churn rate is above or below the overall churn rate
#Red = higher churn than average, Blue = lower churn than average

bar_colors2 = []
for rate in churn_df['ChurnRate']:
    if rate > overall_churn_rate:
        bar_colors2.append('Maroon')   # higher churn than average
    else:
        bar_colors2.append('darkblue') # lower churn than average

# Plot the churn rate for each category with a bar chart, including a dashed line for the overall churn rate.
plt.figure(figsize=(12, 7))
plt.bar(churn_df['Category'], churn_df['ChurnRate'], color=bar_colors2, edgecolor='black')
plt.axhline(y=overall_churn_rate, color='black', linestyle='--',
            label=f'Overall Churn Rate ({overall_churn_rate:.1f}%)')
plt.title('Churn Rate by Category')
plt.ylabel('Churn Rate (%)')
plt.xlabel('Category')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.tight_layout()
plt.show()

# Uses probability outputs instead of binary
prediction_LR = model.predict_proba(testData)[:,1]

In [ ]:
# ==============
# Neural Network
# ==============

import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

#Load the data
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')

#Separate Features (X) and Target (y)
X_train_raw = train_df.drop('Churn', axis=1).values
y_train_raw = train_df['Churn'].values
X_test_raw = test_df.values

#Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

#Convert to Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

class ChurnModel(nn.Module):
    def __init__(self, input_size):
        super(ChurnModel, self).__init__()

        #Layer 1: Input (45 features) -> 64 Hidden Neurons
        self.fc1 = nn.Linear(input_size, 64)

        #Layer 2: 64 Neurons -> 32 Hidden Neurons
        self.fc2 = nn.Linear(64, 32)

        #Layer 3: 32 Neurons -> 1 Output
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        #Pass through Layer 1 and apply ReLU activation
        x = F.relu(self.fc1(x))

        #Pass through Layer 2 and apply ReLU activation
        x = F.relu(self.fc2(x))

        #Output Layer: Use Sigmoid to get a value between 0 and 1
        x = torch.sigmoid(self.fc3(x))

        return x

#Initialize the model using the number of columns (45)
input_dim = X_train_tensor.shape[1]
model = ChurnModel(input_dim)

print(model)

#Define the Loss Function
criterion = nn.BCELoss()

#Define the Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function and Optimizer are ready.")

#Combine X and y into a single Dataset object
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

#Create the DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

print("DataLoader is ready.")

epochs = 10

for epoch in range(epochs):
    model.train() #Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        #Clear the old gradients
        optimizer.zero_grad()

        #Forward Pass: Get predictions
        outputs = model(inputs)

        #Calculate Loss
        loss = criterion(outputs, labels)

        #Backward Pass
        loss.backward()

        #Optimizer Step: Update the weights
        optimizer.step()

        running_loss += loss.item()

    #Print progress after each epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training Complete!")

#Switch to evaluation mode
model.eval()

#No need to track gradients
with torch.no_grad():
    #Pass the entire test set through the model
    test_outputs = model(X_test_tensor)

    #Convert back to a simple NumPy array for easier reading
    probabilities = test_outputs.numpy()

#Look at the first 5 predictions
print("First 5 Churn Probabilities:")
print(probabilities[:5])

#Get the starting ID
start_id = 594194

#Flatten the predictions
probs_flat = probabilities.flatten()

#Create the IDs
test_ids = range(start_id, start_id + len(probs_flat))

#Create the Final DataFrame
submission_df = pd.DataFrame({
    'id': test_ids,
    'Churn': probs_flat
})
